# Capple Chat API - cURL Flow Playground

This notebook helps you test multiple chat flows using raw `curl` calls against `/api/chat` (SSE endpoint).

What is covered:
- Multiple intents: `app_help`, `battery_levels`, `suggest_plan`, `general_chat`
- Ambiguous prompt handling
- Multi-turn conversation history
- Basic auth and validation checks

## 1) Configure Environment

Set your API base URL and a valid Bearer token.

You can paste a Supabase/JWT token you already use for the frontend.

In [3]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [4]:
import json
import shlex
import subprocess
from typing import Any

BASE_URL = "http://localhost:8080"
TOKEN = "eyJhbGciOiJFUzI1NiIsImtpZCI6ImJiYzFmY2Y2LTlkZjEtNDlmMi04MTUyLTBkNjFmYzNiMTZmZCIsInR5cCI6IkpXVCJ9.eyJpc3MiOiJodHRwczovL3FidmJoa21uaWhxa3Vtb3NtbGR4LnN1cGFiYXNlLmNvL2F1dGgvdjEiLCJzdWIiOiJhN2ZkYjY4My01NzMxLTRiM2YtOGM4Zi1iNjRkMzg3NjNmZTYiLCJhdWQiOiJhdXRoZW50aWNhdGVkIiwiZXhwIjoxNzc5NjM4MjIyLCJpYXQiOjE3Nzk2MzQ2MjIsImVtYWlsIjoiZGFuaWVsLnZoeWFAZ21haWwuY29tIiwicGhvbmUiOiIiLCJhcHBfbWV0YWRhdGEiOnsicHJvdmlkZXIiOiJnb29nbGUiLCJwcm92aWRlcnMiOlsiZ29vZ2xlIl19LCJ1c2VyX21ldGFkYXRhIjp7ImF2YXRhcl91cmwiOiJodHRwczovL2xoMy5nb29nbGV1c2VyY29udGVudC5jb20vYS9BQ2c4b2NMcnhqa3lDME01NWYyRTNmWTVRei10aWdJR0h2NW5CbjZtdE1WUnFnQURkbEZwYUE9czk2LWMiLCJlbWFpbCI6ImRhbmllbC52aHlhQGdtYWlsLmNvbSIsImVtYWlsX3ZlcmlmaWVkIjp0cnVlLCJmdWxsX25hbWUiOiJEYW5pZWwgVmFyZ2FzIEhlcnJlcmEiLCJpc3MiOiJodHRwczovL2FjY291bnRzLmdvb2dsZS5jb20iLCJuYW1lIjoiRGFuaWVsIFZhcmdhcyBIZXJyZXJhIiwicGhvbmVfdmVyaWZpZWQiOmZhbHNlLCJwaWN0dXJlIjoiaHR0cHM6Ly9saDMuZ29vZ2xldXNlcmNvbnRlbnQuY29tL2EvQUNnOG9jTHJ4amt5QzBNNTVmMkUzZlk1UXotdGlnSUdIdjVuQm42bXRNVlJxZ0FEZGxGcGFBPXM5Ni1jIiwicHJvdmlkZXJfaWQiOiIxMTc4MzEyODQzNTE2ODI4NDU4NjEiLCJzdWIiOiIxMTc4MzEyODQzNTE2ODI4NDU4NjEifSwicm9sZSI6ImF1dGhlbnRpY2F0ZWQiLCJhYWwiOiJhYWwxIiwiYW1yIjpbeyJtZXRob2QiOiJvYXV0aCIsInRpbWVzdGFtcCI6MTc3ODc4MDA3OH1dLCJzZXNzaW9uX2lkIjoiZTllNWIyY2MtNTMxNi00MDAxLWI5NjgtMTYwOGJkZGQxZmNhIiwiaXNfYW5vbnltb3VzIjpmYWxzZX0.6vEMVAozh1F7lN0awTh4Bbdgu8kaBZWEDwHHGfAhCc4zJjhS7QBZJpk8UmfFcxA7CIkQjcs-N8cGg0i8rddORQ"
CHAT_PATH = "/api/chat"

assert BASE_URL, "BASE_URL is required"
assert CHAT_PATH.startswith('/'), "CHAT_PATH should start with '/'"

## 2) Helper: run cURL and print SSE

`run_chat` sends a request to `/api/chat` with a `message` and `history`, then prints raw SSE lines.

In [5]:
def _build_chat_cmd(message: str, history: list[dict[str, str]], token: str) -> list[str]:
    url = f"{BASE_URL}{CHAT_PATH}"
    payload: dict[str, Any] = {
        "message": message,
        "history": history,
    }
    return [
        "curl",
        "-sS",
        "-N",
        "--no-buffer",
        "-X",
        "POST",
        url,
        "-H",
        "Content-Type: application/json",
        "-H",
        f"Authorization: Bearer {token}",
        "-H",
        "Accept: text/event-stream",
        "--data",
        json.dumps(payload),
    ]


def _redact_cmd(cmd: list[str]) -> list[str]:
    redacted = cmd.copy()
    for i, part in enumerate(redacted):
        if part.startswith("Authorization: Bearer "):
            redacted[i] = "Authorization: Bearer <REDACTED>"
    return redacted


def run_curl(cmd: list[str]) -> tuple[int, str, str]:
    print("$", " ".join(shlex.quote(part) for part in _redact_cmd(cmd)))
    completed = subprocess.run(cmd, capture_output=True, text=True)
    return completed.returncode, completed.stdout, completed.stderr


def parse_sse(text: str) -> list[dict[str, str]]:
    events: list[dict[str, str]] = []
    cur_event = "message"
    cur_id = ""
    cur_data_parts: list[str] = []

    for raw_line in text.splitlines():
        line = raw_line.rstrip("\r")
        if not line:
            if cur_data_parts:
                events.append(
                    {
                        "event": cur_event,
                        "id": cur_id,
                        "data": "\n".join(cur_data_parts),
                    }
                )
            cur_event = "message"
            cur_id = ""
            cur_data_parts = []
            continue

        if line.startswith(":"):
            continue
        if line.startswith("event:"):
            cur_event = line.split(":", 1)[1].strip()
        elif line.startswith("id:"):
            cur_id = line.split(":", 1)[1].strip()
        elif line.startswith("data:"):
            cur_data_parts.append(line.split(":", 1)[1].lstrip())

    if cur_data_parts:
        events.append(
            {
                "event": cur_event,
                "id": cur_id,
                "data": "\n".join(cur_data_parts),
            }
        )

    return events


def run_chat(
    message: str,
    history: list[dict[str, str]] | None = None,
    token: str | None = None,
    stream_debug: bool = True,
    show_parsed_events: bool = True,
) -> list[dict[str, str]]:
    if history is None:
        history = []

    auth_token = token if token is not None else TOKEN
    cmd = _build_chat_cmd(message, history, auth_token)

    if stream_debug:
        print("$", " ".join(shlex.quote(part) for part in _redact_cmd(cmd)))
        print("\n--- raw SSE lines ---")
        proc = subprocess.Popen(
            cmd,
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            text=True,
            bufsize=1,
        )
        assert proc.stdout is not None
        assert proc.stderr is not None

        stdout_lines: list[str] = []
        for line in proc.stdout:
            stdout_lines.append(line)
            print(repr(line.rstrip("\n")))

        stderr_text = proc.stderr.read()
        exit_code = proc.wait()
        raw_text = "".join(stdout_lines)
    else:
        exit_code, raw_text, stderr_text = run_curl(cmd)

    print("\n--- exit code ---")
    print(exit_code)

    if stderr_text:
        print("\n--- stderr ---")
        print(stderr_text)

    events = parse_sse(raw_text)
    if show_parsed_events:
        print("\n--- parsed events ---")
        for idx, event in enumerate(events, start=1):
            print(f"{idx}. event={event['event']} id={event['id']} data={event['data']}")

    return events

## 3) Intent Flow Examples

Run each cell independently to test parser + prompt routing behavior.

In [6]:
# app_help intent
run_chat("How do I create or join a household in Capple?")

$ curl -sS -N --no-buffer -X POST http://localhost:8080/api/chat -H 'Content-Type: application/json' -H 'Authorization: Bearer <REDACTED>' -H 'Accept: text/event-stream' --data '{"message": "How do I create or join a household in Capple?", "history": []}'

--- raw SSE lines ---
': stream of chat updates'
''
'event: message'
'data: "To"'
'id: 1'
''
'event: message'
'data: " create"'
'id: 2'
''
'event: message'
'data: " or"'
'id: 3'
''
'event: message'
'data: " join"'
'id: 4'
''
'event: message'
'data: " a"'
'id: 5'
''
'event: message'
'data: " household"'
'id: 6'
''
'event: message'
'data: " in"'
'id: 7'
''
'event: message'
'data: " Cap"'
'id: 8'
''
'event: message'
'data: "ple"'
'id: 9'
''
'event: message'
'data: ":\\n\\n"'
'id: 10'
''
'event: message'
'data: "##"'
'id: 11'
''
'event: message'
'data: " Create"'
'id: 12'
''
'event: message'
'data: " a"'
'id: 13'
''
'event: message'
'data: " household"'
'id: 14'
''
'event: message'
'data: "\\n"'
'id: 15'
''
'event: message'
'data: "1"'
'

[{'event': 'message', 'id': '1', 'data': '"To"'},
 {'event': 'message', 'id': '2', 'data': '" create"'},
 {'event': 'message', 'id': '3', 'data': '" or"'},
 {'event': 'message', 'id': '4', 'data': '" join"'},
 {'event': 'message', 'id': '5', 'data': '" a"'},
 {'event': 'message', 'id': '6', 'data': '" household"'},
 {'event': 'message', 'id': '7', 'data': '" in"'},
 {'event': 'message', 'id': '8', 'data': '" Cap"'},
 {'event': 'message', 'id': '9', 'data': '"ple"'},
 {'event': 'message', 'id': '10', 'data': '":\\n\\n"'},
 {'event': 'message', 'id': '11', 'data': '"##"'},
 {'event': 'message', 'id': '12', 'data': '" Create"'},
 {'event': 'message', 'id': '13', 'data': '" a"'},
 {'event': 'message', 'id': '14', 'data': '" household"'},
 {'event': 'message', 'id': '15', 'data': '"\\n"'},
 {'event': 'message', 'id': '16', 'data': '"1"'},
 {'event': 'message', 'id': '17', 'data': '"."'},
 {'event': 'message', 'id': '18', 'data': '" Open"'},
 {'event': 'message', 'id': '19', 'data': '" **"'}

In [7]:
# battery_levels intent
run_chat("How is my partner's energy level this week?")

$ curl -sS -N --no-buffer -X POST http://localhost:8080/api/chat -H 'Content-Type: application/json' -H 'Authorization: Bearer <REDACTED>' -H 'Accept: text/event-stream' --data '{"message": "How is my partner'"'"'s energy level this week?", "history": []}'

--- raw SSE lines ---
': stream of chat updates'
''
'event: message'
'data: "I"'
'id: 1'
''
'event: message'
'data: "\\u2019ll"'
'id: 2'
''
'event: message'
'data: " check"'
'id: 3'
''
'event: message'
'data: " the"'
'id: 4'
''
'event: message'
'data: " battery"'
'id: 5'
''
'event: message'
'data: "/"'
'id: 6'
''
'event: message'
'data: "energy"'
'id: 7'
''
'event: message'
'data: " log"'
'id: 8'
''
'event: message'
'data: " for"'
'id: 9'
''
'event: message'
'data: " this"'
'id: 10'
''
'event: message'
'data: " week"'
'id: 11'
''
'event: message'
'data: "."'
'id: 12'
''
'event: message'
'data: "\\n"'
'id: 13'
''
'event: message'
'data: "I"'
'id: 14'
''
'event: message'
'data: " can"'
'id: 15'
''
'event: message'
'data: " check"'
'id

[{'event': 'message', 'id': '1', 'data': '"I"'},
 {'event': 'message', 'id': '2', 'data': '"\\u2019ll"'},
 {'event': 'message', 'id': '3', 'data': '" check"'},
 {'event': 'message', 'id': '4', 'data': '" the"'},
 {'event': 'message', 'id': '5', 'data': '" battery"'},
 {'event': 'message', 'id': '6', 'data': '"/"'},
 {'event': 'message', 'id': '7', 'data': '"energy"'},
 {'event': 'message', 'id': '8', 'data': '" log"'},
 {'event': 'message', 'id': '9', 'data': '" for"'},
 {'event': 'message', 'id': '10', 'data': '" this"'},
 {'event': 'message', 'id': '11', 'data': '" week"'},
 {'event': 'message', 'id': '12', 'data': '"."'},
 {'event': 'message', 'id': '13', 'data': '"\\n"'},
 {'event': 'message', 'id': '14', 'data': '"I"'},
 {'event': 'message', 'id': '15', 'data': '" can"'},
 {'event': 'message', 'id': '16', 'data': '" check"'},
 {'event': 'message', 'id': '17', 'data': '","'},
 {'event': 'message', 'id': '18', 'data': '" but"'},
 {'event': 'message', 'id': '19', 'data': '" I"'},
 {'

In [ ]:
# suggest_plan intent with city explicitly provided
run_chat("Suggest a low-energy date idea in Madrid tonight.")

In [ ]:
# suggest_plan intent without city (should ask for city)
run_chat("What should we do tonight?")

In [ ]:
# ambiguous/short prompt (should clarify intent when confidence is low)
run_chat("help")

## 4) Multi-Turn Conversation History

Use `history` to continue context across turns.

In [ ]:
history = [
    {"role": "user", "content": "What should we do tonight?"},
    {"role": "assistant", "content": "Sure, which city should I use for suggestions?"},
    {"role": "user", "content": "Berlin"},
]

run_chat("Great, give me two options and keep it low-energy.", history=history)

In [ ]:
history = [
    {"role": "user", "content": "How are our battery levels lately?"},
    {"role": "assistant", "content": "You are both stable this week, with slightly lower energy after workdays."},
]

run_chat("Can you summarize that in one sentence?", history=history)

## 5) Validation and Auth Checks

Useful for debugging endpoint behavior quickly.

In [ ]:
# Missing/invalid token
run_chat("hello", token="invalid-token")

In [ ]:
# Raw curl with invalid history type (should return 422)
url = f"{BASE_URL}{CHAT_PATH}"
bad_payload = {"message": "hello", "history": "not-a-list"}
cmd = [
    "curl",
    "-sS",
    "-i",
    "-X",
    "POST",
    url,
    "-H",
    "Content-Type: application/json",
    "-H",
    f"Authorization: Bearer {TOKEN}",
    "--data",
    json.dumps(bad_payload),
]
code, out, err = run_curl(cmd)
print(out if out else "<empty>")
if err:
    print(err)

## 6) Copy-Paste cURL Templates

Use these directly in a terminal outside the notebook if preferred.

In [ ]:
print(
    f"""curl -sS -N -X POST {BASE_URL}{CHAT_PATH} \
  -H 'Content-Type: application/json' \
  -H 'Authorization: Bearer {TOKEN}' \
  --data '{json.dumps({"message": "Suggest a date idea in Berlin", "history": []})}'"""
)